In [3]:
! pip install seqeval
import torch 
import torch.nn as nn
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader 
from torch.nn.utils.rnn import pad_sequence 
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report, f1_score 

In [4]:
df = pd.read_csv('/Users/vydhyamvishnusai/BILSTM_NAMED_ENTITY/ner_dataset.csv', encoding = 'latin1')


In [5]:
df.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


In [6]:
df["Sentence #"] = df["Sentence #"].ffill()

In [7]:
df

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,Sentence: 1,of,IN,O
2,Sentence: 1,demonstrators,NNS,O
3,Sentence: 1,have,VBP,O
4,Sentence: 1,marched,VBN,O
...,...,...,...,...
1048570,Sentence: 47959,they,PRP,O
1048571,Sentence: 47959,responded,VBD,O
1048572,Sentence: 47959,to,TO,O
1048573,Sentence: 47959,the,DT,O


In [8]:
print(f"\nTag distribution:\n{df['Tag'].value_counts()}")


Tag distribution:
Tag
O        887908
B-geo     37644
B-tim     20333
B-org     20143
I-per     17251
B-per     16990
I-org     16784
B-gpe     15870
I-geo      7414
I-tim      6528
B-art       402
B-eve       308
I-art       297
I-eve       253
B-nat       201
I-gpe       198
I-nat        51
Name: count, dtype: int64


# Group into Sentences

In [9]:
sentences = df.groupby("Sentence #")["Word"].apply(list).values 
tag_sequences = df.groupby("Sentence #")["Tag"].apply(list).values 
print(f"Number of sentences: {len(sentences)}")
print(f"\nExample sentence: {sentences[0]}")
print(f"Tags:              {tag_sequences[0]}")
print(f"Sentence length:   {len(sentences[0])}")

lengths = [len(s) for s in sentences]
print(f"\nMin length: {min(lengths)}")
print(f"Max length: {max(lengths)}")
print(f"Mean length: {np.mean(lengths):.1f}")
print(f"95th percentile: {np.percentile(lengths, 95):.0f}")

Number of sentences: 47959

Example sentence: ['Thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the', 'war', 'in', 'Iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'British', 'troops', 'from', 'that', 'country', '.']
Tags:              ['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O', 'O']
Sentence length:   24

Min length: 1
Max length: 104
Mean length: 21.9
95th percentile: 35


# Building Vocabularies

In [10]:
df = df.dropna(subset = ["Word", "Tag"]) 
df["Word"] = df["Word"].astype(str)
sentences = df.groupby("Sentence #")["Word"].apply(list).values 
tag_sequences = df.groupby("Sentence #")["Tag"].apply(list).values 
print(f"Cleaned Setenetces: {len(sentences)}")

Cleaned Setenetces: 47959


In [11]:
word_counts = {}
for sent in sentences:
    for w in sent:
        w_lower = w.lower()
        word_counts[w_lower] = word_counts.get(w_lower, 0) + 1 
word2idx = {"<PAD>": 0, "<UNK>": 1}
for word, count in sorted(word_counts.items()):
    if count >= 2:
        word2idx[word] = len(word2idx)
all_tags = sorted(df["Tag"].unique())
tag2idx = {tag:i for i , tag in enumerate(all_tags)}
idx2tag = {i:tag for tag, i in tag2idx.items()}
print(f"Vocabulary size: {len(word2idx)}")
print(f"Number of tags: {len(all_tags)}: {tag2idx}")

Vocabulary size: 18802
Number of tags: 17: {'B-art': 0, 'B-eve': 1, 'B-geo': 2, 'B-gpe': 3, 'B-nat': 4, 'B-org': 5, 'B-per': 6, 'B-tim': 7, 'I-art': 8, 'I-eve': 9, 'I-geo': 10, 'I-gpe': 11, 'I-nat': 12, 'I-org': 13, 'I-per': 14, 'I-tim': 15, 'O': 16}


# Train-Valid-Test-Split

In [12]:
train_sents, temp_sents, train_tags, temp_tags = train_test_split(sentences, tag_sequences, test_size = 0.2, random_state = 42)
val_sents, test_sents, val_tags, test_tags = train_test_split(temp_sents, temp_tags, test_size = 0.5, random_state = 42)
print(f"Training set size: {len(train_sents)}")
print(f"Validation set size: {len(val_sents)}")
print(f"Test set size: {len(test_sents)}")

Training set size: 38367
Validation set size: 4796
Test set size: 4796


In [13]:
class NERDataset(Dataset):
    def __init__(self, sentences, tags, word2idx, tag2idx):
        self.sentences = sentences 
        self.tags = tags
        self.word2idx = word2idx 
        self.tag2idx = tag2idx
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self,idx):
        x = torch.tensor(
            [self.word2idx.get(w.lower(), self.word2idx["<UNK>"]) for w in self.sentences[idx]],
            dtype = torch.long 
        ) 
        y = torch.tensor(
            [self.tag2idx[t] for t in self.tags[idx]],
            dtype = torch.long 
        )
        return x, y
def collate_fn(batch):
    xs,ys = zip(*batch)
    xs_padded = pad_sequence(xs, batch_first = True, padding_value = 0)
    ys_padded = pad_sequence(ys, batch_first = True, padding_value = -1)
    return xs_padded, ys_padded

train_loader = DataLoader(
    NERDataset(train_sents, train_tags, word2idx, tag2idx),
    batch_size = 32, shuffle = True,
    collate_fn = collate_fn
)
val_loader = DataLoader(
    NERDataset(val_sents, val_tags, word2idx, tag2idx),
    batch_size = 64, shuffle = False, collate_fn = collate_fn 
)
test_loader = DataLoader(
    NERDataset(test_sents, test_tags, word2idx, tag2idx),
    batch_size = 64, shuffle = False, collate_fn = collate_fn 
)
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
x_batch, y_batch = next(iter(train_loader))
print(f"\nBatch shape - X: {x_batch.shape}, Y: {y_batch.shape}")

Train batches: 1199
Validation batches: 75
Test batches: 75

Batch shape - X: torch.Size([32, 45]), Y: torch.Size([32, 45])


# BiLSTM Model

In [17]:
class BiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, tagset_size,  embedding_dim = 128, hidden_dim = 256, num_layers = 2, dropout = 0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx = 0) 
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim // 2 ,# half because bidirectional doubles it 
            bidirectional = True,
            batch_first = True, 
            dropout = dropout  if num_layers > 1 else 0 
        )
        self.hidden2tag = nn.Linear(hidden_dim , tagset_size)
    def forward(self,x):
        # tokens to embeddings
        embeds = self.dropout(self.embedding(x))
        # step 2 embeddings -> bilstm (contextual repersentations)
        lstm_out, _ = self.lstm(embeds)
        lstm_out = self.dropout(lstm_out)
        # bilstm output -> Tag scores (logits)
        logits = self.hidden2tag(lstm_out)
        return logits 
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = BiLSTM_NER(len(word2idx), len(tag2idx)).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Total parameters: {total_params}")
print(f"Model architecture:\n{model}")




        


Device: mps
Total parameters: 2675217
Model architecture:
BiLSTM_NER(
  (embedding): Embedding(18802, 128, padding_idx=0)
  (dropout): Dropout(p=0.3, inplace=False)
  (lstm): LSTM(128, 128, batch_first=True, dropout=0.3, bidirectional=True)
  (hidden2tag): Linear(in_features=256, out_features=17, bias=True)
)


/opt/anaconda3/envs/mlproject/lib/python3.14/site-packages/torch/nn/modules/rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index = -1)
optimizer = optim.Adam(model.parameters(), lr = 0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience = 2, factor = 0.5)
best_val_f1 = 0 
patience_counter = 0 
history = {"train_loss": [], "val_f1": []}
for epoch in range(20):
    model.train()
    total_loss = 0
    for x,y in train_loader: 
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, len(tag2idx)), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)
    model.eval()
    

        
        

In [21]:
train_loader